# Naive Bayes Classifier

## 1. Bayes' Theorem

## 1.1 Probability

**Definition (Probability space).** A probability space is a triple $(\Omega, \mathcal{A}, P)$, where:

- $\Omega$ is the **sample space**: the set of all possible outcomes $\omega$ of an experiment.
- $\mathcal{A} \subseteq 2^{\Omega}$ is a **$\sigma$-algebra** on $\Omega$:
a collection of subsets of $\Omega$ (called **events**) such that
  1. $\Omega \in \mathcal{A}$
  2. if $A \in \mathcal{A}$, then $A^c = \Omega \setminus A \in \mathcal{A}$ (closed under complement)
  3. if $A_1, A_2, \ldots \in \mathcal{A}$, then $\bigcup_{i=1}^{\infty} A_i \in \mathcal{A}$
(closed under countable unions)
- $P : \mathcal{A} \to [0, 1]$ is a **probability measure** satisfying the Kolmogorov axioms:
  1. $P(A) \geq 0$ for every $A \in \mathcal{A}$
  2. $P(\Omega) = 1$
  3. for any pairwise disjoint events
  $A_1, A_2, \ldots \in \mathcal{A}$ (i.e. $A_i \cap A_j = \emptyset$ for $i \neq j$):
  $$
  P\left(\bigcup_{i=1}^{\infty} A_i\right) = \sum_{i=1}^{\infty} P(A_i)
  $$

The number $P(A)$ is called the **probability** of the event $A$.

## 1.2 Classical Probability

$$
P(E) = \frac{\text{Number of favorable outcomes}}{\text{Total number of outcomes}}
$$

## 1.3 Conditional Probability

**Definition (Conditional probability).** Let $(\Omega, \mathcal{A}, P)$ be a probability space
and let $A, B \in \mathcal{A}$ be events with $P(B) > 0$.
The **conditional probability** of $A$ given $B$ is

$$
P(A \mid B) = \frac{P(A \cap B)}{P(B)}
$$

$P(A \mid B)$ is the probability that $A$ occurs, given that $B$ has occurred.
Knowing that $B$ occurred shrinks the sample space from $\Omega$ to $B$.

![Conditional Probability](../../images/cond-prob.svg)

## 1.4 Proof of Bayes' Theorem

Let $H, E \in \mathcal{A}$ with $P(H) > 0$ and $P(E) > 0$. From the definition of conditional probability we get:

$$
P(H \cap E) = P(H \mid E) \cdot P(E)
$$

$$
P(E \cap H) = P(E \mid H) \cdot P(H)
$$

Since intersection is commutative, $H \cap E = E \cap H$, so the left-hand sides are equal:

$$
P(H \mid E) \cdot P(E) = P(E \mid H) \cdot P(H)
$$

Dividing both sides by $P(E) > 0$:

$$
P(H \mid E) = \frac{P(E \mid H) \cdot P(H)}{P(E)}
$$

## 1.5 Bayes' Theorem

$$
P(H \mid E) = \frac{P(E \mid H) \, P(H)}{P(E)}
$$

where:

- $P(H \mid E)$ — **posterior**: the probability of the hypothesis $H$ after observing the evidence $E$
- $P(E \mid H)$ — **likelihood**: the probability of observing the evidence $E$ if the hypothesis $H$ is true
- $P(H)$ — **prior**: the probability of the hypothesis $H$ before observing any evidence
- $P(E)$ — **evidence** (marginal likelihood): the total probability of observing the evidence $E$


## 1.6 Bayes' Theorem in Naive Bayes Classifier

In classification the evidence $E$ is not a single event but a combination
of observed features (attributes) $x_1, x_2, \ldots, x_n$. All of them are observed at the same time,
so the evidence is their intersection:

$$
E = x_1 \cap x_2 \cap \ldots \cap x_n
$$

Substituting this into Bayes' theorem:

$$
P(H \mid x_1 \cap x_2 \cap \ldots \cap x_n) =
\frac{P(x_1 \cap x_2 \cap \ldots \cap x_n \mid H) \, P(H)}{P(x_1 \cap x_2 \cap \ldots \cap x_n)}
$$

It is rare to write the intersection symbol $\cap$ explicitly.
By convention, a comma-separated list of events means their intersection,
i.e. $P(x_1, x_2, \ldots, x_n) = P(x_1 \cap x_2 \cap \ldots \cap x_n)$.
So Bayes' theorem is usually written as:

$$
P(H \mid x_1, x_2, \ldots, x_n) = \frac{P(x_1, x_2, \ldots, x_n \mid H) \, P(H)}{P(x_1, x_2, \ldots, x_n)}
$$

# 2. Why is Bayes naive?

## 2.1 Lemma (Product Rule for Conditional Probability)

**Lemma.** Let $A, B, C \in \mathcal{A}$ be events with $P(B, C) > 0$. Then

$$
P(A, B \mid C) = P(A \mid B, C) \cdot P(B \mid C)
$$

Note that $P(C) \geq P(B, C) > 0$, so all three conditional probabilities are defined.

**Proof.** By the definition of conditional probability, multiplying and dividing by $P(B, C) > 0$:

$$
\begin{aligned}
P(A, B \mid C) &= \frac{P(A, B, C)}{P(C)} \\[6pt]
&= \frac{P(A, B, C)}{P(B, C)} \cdot \frac{P(B, C)}{P(C)} \\[6pt]
&= P(A \mid B, C) \cdot P(B \mid C)
\end{aligned}
$$

## 2.2 Conditional Independence

Let $A, B, C \in \mathcal{A}$ be events with $P(C) > 0$ and $P(B, C) > 0$.
There are two equivalent definitions of **conditional independence** of $A$ and $B$ given $C$:

**Definition 1.**
$$
P(A, B \mid C) = P(A \mid C) \cdot P(B \mid C)
$$

**Definition 2.**
$$
P(A \mid B, C) = P(A \mid C)
$$

Definition 2 says that once $C$ is known, also knowing $B$ gives no extra information about $A$.

**Claim.** Definition 1 $\iff$ Definition 2.

In both directions we use the Lemma (2.1):

$$
P(A, B \mid C) = P(A \mid B, C) \cdot P(B \mid C)
$$

**Proof (1 $\Rightarrow$ 2).** Assume $P(A, B \mid C) = P(A \mid C) \cdot P(B \mid C)$. Then

$$
\begin{aligned}
P(A \mid B, C) &= \frac{P(A, B \mid C)}{P(B \mid C)} \\[6pt]
&= \frac{P(A \mid C) \cdot P(B \mid C)}{P(B \mid C)} \\[6pt]
&= P(A \mid C)
\end{aligned}
$$

**Proof (2 $\Rightarrow$ 1).** Assume $P(A \mid B, C) = P(A \mid C)$. Then, by the Lemma,

$$
\begin{aligned}
P(A, B \mid C) &= P(A \mid B, C) \cdot P(B \mid C) \\[6pt]
&= P(A \mid C) \cdot P(B \mid C)
\end{aligned}
$$

## 2.3 Mutual Conditional Independence

Conditional independence of two events (2.2) extends to any number of events.

Let $A_1, A_2, \ldots, A_n, C \in \mathcal{A}$ be events with $P(C) > 0$.
For a set of indices $S \subseteq \{1, \ldots, n\}$ write $A_S$
for the intersection of the events with indices in $S$
(by convention $A_\emptyset = \Omega$):

$$
A_S = \bigcap_{j \in S} A_j
$$

so that, for example, $P(A_{\{1, 3\}} \mid C) = P(A_1, A_3 \mid C)$.
Assume also that $P(A_S, C) > 0$ for every $S$, so that all conditional probabilities below are defined.
There are two equivalent definitions of **mutual conditional independence** of $A_1, \ldots, A_n$ given $C$:

**Definition 1.** For every nonempty $S \subseteq \{1, \ldots, n\}$:
$$
P(A_S \mid C) = \prod_{j \in S} P(A_j \mid C)
$$

**Definition 2.** For every $i \in \{1, \ldots, n\}$ and every $S \subseteq \{1, \ldots, n\} \setminus \{i\}$:
$$
P(A_i \mid A_S, C) = P(A_i \mid C)
$$

Definition 2 says that once $C$ is known, also knowing any combination
of the other events gives no extra information about $A_i$.

For $n = 2$ both definitions reduce to the ones in 2.2.
Note that Definition 1 must hold for **every** subset $S$, not only for pairs:
pairwise conditional independence of $A_1, \ldots, A_n$ given $C$ does not imply mutual conditional independence.

**Claim.** Definition 1 $\iff$ Definition 2.

**Proof (1 $\Rightarrow$ 2).** Assume Definition 1. Let $i \in \{1, \ldots, n\}$
and $S \subseteq \{1, \ldots, n\} \setminus \{i\}$.
If $S = \emptyset$, then $A_S = \Omega$ and the statement is trivial.
Otherwise, the Lemma (2.1) with $A = A_i$ and $B = A_S$ gives

$$
P(A_i, A_S \mid C) = P(A_i \mid A_S, C) \cdot P(A_S \mid C)
$$

Then, using $A_i \cap A_S = A_{S \cup \{i\}}$ and Definition 1 for both $S \cup \{i\}$ and $S$:

$$
\begin{aligned}
P(A_i \mid A_S, C) &= \frac{P(A_i, A_S \mid C)}{P(A_S \mid C)} \\[6pt]
&= \frac{P(A_{S \cup \{i\}} \mid C)}{P(A_S \mid C)} \\[6pt]
&= \frac{P(A_i \mid C) \cdot \prod_{j \in S} P(A_j \mid C)}{\prod_{j \in S} P(A_j \mid C)} \\[6pt]
&= P(A_i \mid C)
\end{aligned}
$$

**Proof (2 $\Rightarrow$ 1).** Assume Definition 2. Let $S = \{i_1, i_2, \ldots, i_k\}$ be nonempty.
Applying the Lemma (2.1) with $A = A_{i_1}$ and $B = A_{i_2} \cap \ldots \cap A_{i_k}$,
then again with $A = A_{i_2}$ and $B = A_{i_3} \cap \ldots \cap A_{i_k}$, and so on (the **chain rule**):

$$
\begin{aligned}
P(A_S \mid C) &= P(A_{i_1}, A_{i_2}, \ldots, A_{i_k} \mid C) \\[6pt]
&= P(A_{i_1} \mid A_{i_2}, \ldots, A_{i_k}, C) \cdot P(A_{i_2}, \ldots, A_{i_k} \mid C) \\[6pt]
&= P(A_{i_1} \mid A_{i_2}, \ldots, A_{i_k}, C) \cdot P(A_{i_2} \mid A_{i_3}, \ldots, A_{i_k}, C) \cdots P(A_{i_k} \mid C) \\[6pt]
&= P(A_{i_1} \mid C) \cdot P(A_{i_2} \mid C) \cdots P(A_{i_k} \mid C) \\[6pt]
&= \prod_{j \in S} P(A_j \mid C)
\end{aligned}
$$

where in the fourth line we applied Definition 2 to each factor.

In Naive Bayes the events are the features $x_1, \ldots, x_n$ and $C$ is the class $C_k$.
The "naive" assumption is exactly mutual conditional independence of the features given the class:
$P(x_i \mid x_{i+1}, \ldots, x_n, C_k) = P(x_i \mid C_k)$.

## 2.4 Bayes' Theorem + Mutual Conditional Independence

Let $C_1, C_2, \ldots, C_K \in \mathcal{A}$ be the **classes**: events with
$C_i \cap C_j = \emptyset$ for $i \neq j$, $\bigcup_{k=1}^{K} C_k = \Omega$ and $P(C_k) > 0$ for every $k$.
Let $x_1, x_2, \ldots, x_n \in \mathcal{A}$ be the observed **features** of the instance we want to classify,
with $P(x_1, x_2, \ldots, x_n) > 0$.

**Step 1 (Bayes' theorem).** Taking the hypothesis $H = C_k$ in 1.6:

$$
P(C_k \mid x_1, x_2, \ldots, x_n) = \frac{P(x_1, x_2, \ldots, x_n \mid C_k) \, P(C_k)}{P(x_1, x_2, \ldots, x_n)}
$$

**Step 2 (the naive assumption).** Assume that the features $x_1, \ldots, x_n$ are
mutually conditionally independent given each class $C_k$ (2.3).
By Definition 1 with $S = \{1, \ldots, n\}$, the likelihood factorizes:

$$
P(x_1, x_2, \ldots, x_n \mid C_k) = \prod_{i=1}^{n} P(x_i \mid C_k)
$$

Equivalently, by the chain rule and Definition 2:

$$
\begin{aligned}
P(x_1, x_2, \ldots, x_n \mid C_k) &= P(x_1 \mid x_2, \ldots, x_n, C_k) \cdot P(x_2 \mid x_3, \ldots, x_n, C_k) \cdots P(x_n \mid C_k) \\[6pt]
&= P(x_1 \mid C_k) \cdot P(x_2 \mid C_k) \cdots P(x_n \mid C_k)
\end{aligned}
$$

**Step 3 (substitute).** Putting the factorized likelihood into Bayes' theorem:

$$
P(C_k \mid x_1, x_2, \ldots, x_n) = \frac{P(C_k) \prod_{i=1}^{n} P(x_i \mid C_k)}{P(x_1, x_2, \ldots, x_n)}
$$

# 3. Naive Bayes Classifier

## 3.1 The Evidence Is a Constant

From 2.4, under the naive assumption the posterior of each class $C_k$ is

$$
P(C_k \mid x_1, x_2, \ldots, x_n) = \frac{P(C_k) \prod_{i=1}^{n} P(x_i \mid C_k)}{P(x_1, x_2, \ldots, x_n)}
$$

The denominator does not depend on $k$: it is the same for every class.
Since the classes form a partition of $\Omega$, by the law of total probability and the factorization from 2.4 it equals

$$
P(x_1, x_2, \ldots, x_n) = \sum_{k=1}^{K} P(x_1, \ldots, x_n \mid C_k) \, P(C_k)
= \sum_{k=1}^{K} P(C_k) \prod_{i=1}^{n} P(x_i \mid C_k)
$$

so it only scales the posteriors so that they sum to $1$. Hence

$$
P(C_k \mid x_1, x_2, \ldots, x_n) \propto P(C_k) \prod_{i=1}^{n} P(x_i \mid C_k)
$$

where $\propto$ means "proportional to" (equal up to a factor that does not depend on $k$).

## 3.2 The Decision Rule

The **Naive Bayes classifier** assigns the class
with the largest posterior probability, the **maximum a posteriori (MAP)** decision rule:

$$
\hat{y} = \underset{k \in \{1, \ldots, K\}}{\operatorname{argmax}} \; P(C_k) \prod_{i=1}^{n} P(x_i \mid C_k)
$$

By 3.1 we can drop the evidence, since dividing every class by the same positive constant does not change which one is the largest.

To use the classifier we only need to estimate from the training data:

- the **priors** $P(C_k)$ for every class
- the **likelihoods** $P(x_i \mid C_k)$ for every feature value and every class

# 4. Tennis Example

In [1]:
import polars as pl

df = pl.read_csv("../../demos/naive-bayes/play_tennis.csv").drop("day")

target = "play"
features = [column for column in df.columns if column != target]

print(f"features: {features}")
print(f"target: {target}")
df

features: ['outlook', 'temp', 'humidity', 'wind']
target: play


outlook,temp,humidity,wind,play
str,str,str,str,str
"""Sunny""","""Hot""","""High""","""Weak""","""No"""
"""Sunny""","""Hot""","""High""","""Strong""","""No"""
"""Overcast""","""Hot""","""High""","""Weak""","""Yes"""
"""Rain""","""Mild""","""High""","""Weak""","""Yes"""
"""Rain""","""Cool""","""Normal""","""Weak""","""Yes"""
…,…,…,…,…
"""Rain""","""Mild""","""Normal""","""Weak""","""Yes"""
"""Sunny""","""Mild""","""Normal""","""Strong""","""Yes"""
"""Overcast""","""Mild""","""High""","""Strong""","""Yes"""


In [2]:
class_counts = df.group_by(target).len(name="class_count").sort(target)

priors = class_counts.with_columns(
    fraction=pl.format("{}/{}", pl.col("class_count"), pl.lit(df.height)),
    prior=pl.col("class_count") / df.height,
)

priors

play,class_count,fraction,prior
str,u32,str,f64
"""No""",5,"""5/14""",0.357143
"""Yes""",9,"""9/14""",0.642857


In [5]:
likelihoods = {}

for feature in features:
    pairs = df.select(feature).unique().join(class_counts, how="cross")
    counts = df.group_by(feature, target).len(name="count")

    likelihoods[feature] = (
        pairs.join(counts, on=[feature, target], how="left")
        .with_columns(pl.col("count").fill_null(0))
        .with_columns(
            fraction=pl.format("{}/{}", pl.col("count"), pl.col("class_count")),
            likelihood=pl.col("count") / pl.col("class_count"),
        )
        .select(feature, target, "fraction", "likelihood")
        .sort(target, feature)
    )
    display(likelihoods[feature])

outlook,play,fraction,likelihood
str,str,str,f64
"""Overcast""","""No""","""0/5""",0.0
"""Rain""","""No""","""2/5""",0.4
"""Sunny""","""No""","""3/5""",0.6
"""Overcast""","""Yes""","""4/9""",0.444444
"""Rain""","""Yes""","""3/9""",0.333333
"""Sunny""","""Yes""","""2/9""",0.222222


temp,play,fraction,likelihood
str,str,str,f64
"""Cool""","""No""","""1/5""",0.2
"""Hot""","""No""","""2/5""",0.4
"""Mild""","""No""","""2/5""",0.4
"""Cool""","""Yes""","""3/9""",0.333333
"""Hot""","""Yes""","""2/9""",0.222222
"""Mild""","""Yes""","""4/9""",0.444444


humidity,play,fraction,likelihood
str,str,str,f64
"""High""","""No""","""4/5""",0.8
"""Normal""","""No""","""1/5""",0.2
"""High""","""Yes""","""3/9""",0.333333
"""Normal""","""Yes""","""6/9""",0.666667


wind,play,fraction,likelihood
str,str,str,f64
"""Strong""","""No""","""3/5""",0.6
"""Weak""","""No""","""2/5""",0.4
"""Strong""","""Yes""","""3/9""",0.333333
"""Weak""","""Yes""","""6/9""",0.666667
